# 4단계: 운영 비율형 변수 추가

## 이 단계의 핵심 변화
야근 승인건수 등을 절대값이 아니라 비율형 신호로도 반영한 단계입니다.

## 왜 점수가 좋아졌나
규모 대비 의미를 보게 되면서 절대값보다 더 안정적인 설명력을 얻었습니다.

## 안내
이 파일은 다운로드 오류를 피하기 위해 새 이름으로 다시 만든 버전입니다.
코드 셀 안에는 기존 주석이 남아 있거나, 원본 설명 흐름을 유지합니다.


# 제출용 노트북
## v7 split-features + 중식만 overtime_per_working 추가

구성:
- **중식**: 현재 최고 분리 피처 + `overtime_per_working`
- **석식**: 현재 최고 분리 피처 baseline 그대로

핵심 구조:
- 참여율 예측 후 식수 인원으로 환산
- 중식/석식 피처 분리 유지

출력 파일:
- `submission_v7_split_lunch_plus_overtime_per_working.csv`

In [ ]:
# 필요시 한 번만 설치
# !pip install xgboost scikit-learn

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from xgboost import XGBRegressor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# ---------------------------
# 1. 파일 경로 자동 탐색
# ---------------------------
train_candidates = [
    r"C:\Users\yuzhd\github\team\data\made\train_median.csv",
    r"/mnt/c/Users/yuzhd/github/team/data/made/train_median.csv",
    r"/mnt/data/train_median.csv",
    r"./train_median.csv",
]

test_candidates = [
    r"C:\Users\yuzhd\github\team\data\test.csv",
    r"C:\Users\yuzhd\github\team\data\made\test.csv",
    r"/mnt/c/Users/yuzhd/github/team/data/test.csv",
    r"/mnt/c/Users/yuzhd/github/team/data/made/test.csv",
    r"/mnt/data/test.csv",
    r"./test.csv",
]

sample_sub_candidates = [
    r"C:\Users\yuzhd\github\team\data\sample_submission.csv",
    r"C:\Users\yuzhd\github\team\data\made\sample_submission.csv",
    r"/mnt/c/Users/yuzhd/github/team/data/sample_submission.csv",
    r"/mnt/c/Users/yuzhd/github/team/data/made/sample_submission.csv",
    r"/mnt/data/sample_submission.csv",
    r"./sample_submission.csv",
]

def find_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

train_path = find_existing_path(train_candidates)
test_path = find_existing_path(test_candidates)
sample_sub_path = find_existing_path(sample_sub_candidates)

print("train path:", train_path)
print("test path:", test_path)
print("sample_submission path:", sample_sub_path)

if train_path is None:
    raise FileNotFoundError("train_median.csv 파일을 찾지 못했습니다.")
if test_path is None:
    raise FileNotFoundError("test.csv 파일을 찾지 못했습니다.")
if sample_sub_path is None:
    raise FileNotFoundError("sample_submission.csv 파일을 찾지 못했습니다.")

In [ ]:
# ---------------------------
# 2. 데이터 로드
# ---------------------------
train = pd.read_csv(train_path, encoding="utf-8-sig")
test = pd.read_csv(test_path, encoding="utf-8-sig")
sample_submission = pd.read_csv(sample_sub_path, encoding="utf-8-sig")

train["일자"] = pd.to_datetime(train["일자"])
test["일자"] = pd.to_datetime(test["일자"])

display(train.head())
display(test.head())
display(sample_submission.head())

In [ ]:
# ---------------------------
# 3. 피처 생성
# ---------------------------
def add_features(df):
    df = df.copy()

    df["월"] = df["일자"].dt.month
    df["일"] = df["일자"].dt.day
    df["요일"] = df["일자"].dt.weekday

    df["식사가능자수"] = (
        df["본사정원수"]
        - df["본사휴가자수"]
        - df["본사출장자수"]
        - df["현본사소속재택근무자수"]
    ).clip(lower=1)

    month_end = df["일자"] + pd.offsets.MonthEnd(0)
    df["days_to_month_end"] = (month_end - df["일자"]).dt.days
    df["is_month_end_part"] = (df["days_to_month_end"] <= 5).astype(int)
    df["is_wed"] = (df["요일"] == 2).astype(int)
    df["is_fri"] = (df["요일"] == 4).astype(int)

    df["has_overtime"] = (df["본사시간외근무명령서승인건수"] > 0).astype(int)
    df["log_overtime"] = np.log1p(df["본사시간외근무명령서승인건수"])
    df["overtime_per_working"] = df["본사시간외근무명령서승인건수"] / (df["식사가능자수"] + 1)

    return df

train = add_features(train)
test = add_features(test)

train["중식참여율"] = (train["중식계"] / train["식사가능자수"]).clip(lower=0, upper=1.5)
train["석식참여율"] = (train["석식계"] / train["식사가능자수"]).clip(lower=0, upper=1.5)

display(train[["일자","식사가능자수","중식참여율","석식참여율","overtime_per_working"]].head())

In [ ]:
# ---------------------------
# 4. 중식/석식 피처 분리
# ---------------------------
# 중식: baseline + overtime_per_working 추가
lunch_features = [
    "월",
    "일",
    "요일",
    "식사가능자수",
    "본사출장자수",
    "본사시간외근무명령서승인건수",
    "is_fri",
    "days_to_month_end",
    "is_month_end_part",
    "overtime_per_working",
]

# 석식: baseline 그대로
dinner_features = [
    "월",
    "일",
    "요일",
    "식사가능자수",
    "본사출장자수",
    "본사시간외근무명령서승인건수",
    "is_wed",
    "has_overtime",
    "log_overtime",
]

X_train_lunch = train[lunch_features]
X_test_lunch = test[lunch_features]
y_lunch = train["중식참여율"]

X_train_dinner = train[dinner_features]
X_test_dinner = test[dinner_features]
y_dinner = train["석식참여율"]

print("중식 피처:", lunch_features)
print("석식 피처:", dinner_features)

In [ ]:
# ---------------------------
# 5. 모델 학습
# ---------------------------
xgb_lunch = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42
)

xgb_dinner = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42
)

xgb_lunch.fit(X_train_lunch, y_lunch)
xgb_dinner.fit(X_train_dinner, y_dinner)

pred_lunch_ratio = np.clip(xgb_lunch.predict(X_test_lunch), 0, 1.5)
pred_dinner_ratio = np.clip(xgb_dinner.predict(X_test_dinner), 0, 1.5)

In [ ]:
# ---------------------------
# 6. 식수 인원으로 환산
# ---------------------------
pred_lunch_count = pred_lunch_ratio * test["식사가능자수"].values
pred_dinner_count = pred_dinner_ratio * test["식사가능자수"].values

pred_lunch_count = np.clip(pred_lunch_count, 0, None)
pred_dinner_count = np.clip(pred_dinner_count, 0, None)

print("중식 식수 예측 평균:", pred_lunch_count.mean())
print("석식 식수 예측 평균:", pred_dinner_count.mean())

In [ ]:
# ---------------------------
# 7. 제출 파일 생성
# ---------------------------
submission = sample_submission.copy()

if "중식계" in submission.columns and "석식계" in submission.columns:
    lunch_col = "중식계"
    dinner_col = "석식계"
else:
    numeric_like_cols = [c for c in submission.columns if c != submission.columns[0]]
    if len(numeric_like_cols) >= 2:
        lunch_col, dinner_col = numeric_like_cols[:2]
    else:
        raise ValueError("sample_submission 컬럼을 자동으로 해석하지 못했습니다.")

submission[lunch_col] = pred_lunch_count
submission[dinner_col] = pred_dinner_count

save_name = "submission_v7_split_lunch_plus_overtime_per_working.csv"
submission.to_csv(save_name, index=False, encoding="utf-8-sig")

print("저장 완료:", save_name)
display(submission.head())